# Tugas Analisis Algoritma: Kruskal, Prim, dan Borůvka (Minimum Spanning Tree)

**Mata kuliah:** Algoritma Komputasi, Magister Teknik Informatika
**Mahasiswa:** Muhammad Alif Qadri (D082261018)
**Algoritma:** No 10, Kruskal dan Prim (MST). Pembanding ketiga: Borůvka.

Notebook ini adalah **Tugas 2 (kode program dan eksperimen)** dalam satu berkas, versi yang berjalan di komputer lokal. Isinya: implementasi tiga algoritma, uji kasus kecil yang dicek dengan hitungan tangan, pembuat graf acak, eksperimen waktu, dan grafik dengan kurva teori.

---
## README

### Pustaka yang dipakai
| Pustaka | Dipakai untuk | Catatan |
|---|---|---|
| `random`, `time`, `math`, `heapq`, `csv`, `os`, `platform`, `statistics` | Pembuat graf, pengukuran, antrean prioritas (alat bantu Prim) | Bawaan Python |
| `pandas`, `numpy` | Ringkasan data, rata-rata dan simpangan baku | Ada di `requirements.txt` |
| `matplotlib` | Grafik | Ada di `requirements.txt` |
| `networkx` | **Hanya verifikasi** (Sel 10), tidak dipakai di dalam algoritma | Opsional, ada di `requirements.txt` |

### Cara menjalankan
**Versi lokal (Windows):** klik dua kali `jalankan_notebook.bat` di folder `CODING`. Berkas itu membuat lingkungan Python, memasang pustaka dari `requirements.txt` (sekali saja, butuh internet), lalu membuka notebook ini. Setelah terbuka, pilih **Run > Run All Cells**.
**Cara lain:** buka notebook ini di VS Code atau Jupyter dengan kernel Python yang sudah memasang `requirements.txt`, lalu jalankan semua sel. Pastikan notebook dijalankan dari foldernya sendiri (`CODING/Eksperimen`).
Folder `hasil/` dan `grafik/` dibuat otomatis **di samping notebook ini**, tanpa Google Drive.
Kode di notebook ini sama dengan file `.py` di folder `Kruskal`, `Prim`, dan `Borůvka` (Sel 10 mencocokkannya).
Untuk uji coba cepat (sekitar 15 detik), ubah `MODE_CEPAT = True` di Sel 1.

### Contoh keluaran (Sel 7, uji kasus kecil)
```
[LOLOS] Kasus 1: 4 kota biasa (satu jalur ditolak) | total = 7 | Kruskal, Prim, Borůvka sama
[LOLOS] Kasus 2: segitiga 3 kota | total = 3 | Kruskal, Prim, Borůvka sama
[LOLOS] Kasus 3: bobot kembar (total dan himpunan sisi) | total = 6 | Kruskal, Prim, Borůvka sama
[LOLOS] Kasus 4: graf tak terhubung (harus ValueError) | ketiganya melempar ValueError

Semua 4 kasus uji lolos pada ketiga algoritma (3 kasus graf terhubung dan 1 kasus galat).
```

### Isi folder hasil
| Berkas | Isi |
|---|---|
| `hasil/eksperimen_mentah.csv` | Waktu setiap ulangan (data mentah) |
| `hasil/ringkasan.csv` | Rata-rata dan simpangan baku per ukuran |
| `hasil/rasio_terhadap_teori.csv` | Pemeriksaan kesesuaian dengan kurva teori |
| `grafik/waktu_<skenario>.png` | Grafik waktu, titik ukur disandingkan dengan kurva teori |

### Peta syarat tugas ke sel
| Syarat | Sel |
|---|---|
| Kruskal dan Prim ditulis sendiri, tanpa pustaka MST sebagai inti | 4, 5 |
| Pembanding ketiga (Borůvka) | 6 |
| Fungsi atau kelas dengan komentar | 4 sampai 6 |
| Minimal 3 kasus uji kecil yang bisa dicek manual | 7 |
| Pembuat graf acak, *seed* tetap, terhubung, jarang dan padat | 8 dan 9 |
| Minimal 5 ukuran, minimal 5 ulangan, rata-rata dan simpangan baku, CSV | 11 sampai 13 |
| Grafik waktu dengan kurva teori | 14 dan 15 |
| Banyak skenario | 1 (tiga skenario) |
| Pemakaian AI dilaporkan, sumber kode dicantumkan | 17 |

## Sel 1. Pengaturan (ubah angka di sini bila perlu)
Semua angka penting ada di sini supaya mudah diubah. Hal yang perlu diingat:
- **`SEED`** membuat graf acak selalu sama setiap dijalankan, sehingga percobaan bisa diulang oleh orang lain.
- **Tiga skenario** (dosen menyarankan banyak skenario): graf jarang, graf padat, dan graf jarang dengan bobot kembar.
- Graf padat tumbuh seperti n², jadi ukurannya sengaja jauh lebih kecil daripada graf jarang. Ukuran dibuat cukup besar supaya waktu tidak terlalu singkat dan pengukuran tidak berisik.

In [ ]:
# ===== SEL 1: PENGATURAN =====
SEED = 2026                    # angka awal pengacak (tetap, supaya hasil bisa diulang)
ULANGAN = 5                    # pengulangan pengukuran per ukuran (minimal 5)
ULANG_GRAF_BARU = False        # False: 5 ulangan memakai graf yang sama (mengukur gangguan waktu)
                               # True : tiap ulangan memakai graf berbeda (seed + nomor ulangan)
BOBOT_MAKS_BIASA = 1_000_000   # bobot acak 1..angka ini, bobot kembar jarang terjadi
BOBOT_MAKS_KEMBAR = 5          # bobot acak 1..5, bobot kembar sangat banyak
MODE_CEPAT = False             # True: ukuran kecil untuk uji coba cepat

SKENARIO = {
    "jarang": {
        "keterangan": "graf jarang, m = 3n",
        "ukuran": [4000, 8000, 16000, 32000, 64000],
        "jumlah_sisi": lambda n: 3 * n,
        "bobot_maks": BOBOT_MAKS_BIASA,
    },
    "padat": {
        "keterangan": "graf padat, m = n(n-1)/4 (separuh dari jumlah sisi maksimum)",
        "ukuran": [200, 400, 600, 800, 1000],
        "jumlah_sisi": lambda n: n * (n - 1) // 4,
        "bobot_maks": BOBOT_MAKS_BIASA,
    },
    "jarang_kembar": {
        "keterangan": "graf jarang, m = 3n, bobot 1..5 (banyak bobot kembar)",
        "ukuran": [4000, 8000, 16000, 32000, 64000],
        "jumlah_sisi": lambda n: 3 * n,
        "bobot_maks": BOBOT_MAKS_KEMBAR,
    },
}

if MODE_CEPAT:  # ukuran kecil, hanya untuk mencoba apakah semua sel berjalan
    SKENARIO["jarang"]["ukuran"] = [100, 200, 400, 800, 1600]
    SKENARIO["padat"]["ukuran"] = [30, 50, 70, 90, 110]
    SKENARIO["jarang_kembar"]["ukuran"] = [100, 200, 400, 800, 1600]

for nama, s in SKENARIO.items():
    print(f"{nama:14s}: {s['keterangan']}; ukuran n = {s['ukuran']}")
print(f"Ulangan per ukuran: {ULANGAN} | seed: {SEED}")

## Sel 2. Buat folder hasil otomatis (versi lokal)
Folder `hasil/` (CSV) dan `grafik/` (gambar) dibuat otomatis **di folder tempat notebook ini dijalankan** (`CODING/Eksperimen`). Tidak perlu Google Drive. Alamat lengkapnya dicetak di bawah supaya mudah ditemukan.

In [ ]:
# ===== SEL 2: BUAT FOLDER HASIL OTOMATIS (VERSI LOKAL) =====
import os

FOLDER_DASAR = os.getcwd()  # folder tempat notebook ini dijalankan
FOLDER_HASIL = os.path.join(FOLDER_DASAR, "hasil")
FOLDER_GRAFIK = os.path.join(FOLDER_DASAR, "grafik")
for folder in (FOLDER_HASIL, FOLDER_GRAFIK):
    os.makedirs(folder, exist_ok=True)
    print("Folder siap:", folder)

## Sel 3. Impor pustaka dan catat lingkungan uji
Informasi lingkungan ini dipakai untuk bagian "lingkungan uji" di Bab 4 laporan.

In [ ]:
# ===== SEL 3: IMPOR PUSTAKA DAN LINGKUNGAN UJI =====
import csv
import heapq
import math
import platform
import random
import statistics
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python      :", sys.version.split()[0])
print("Sistem      :", platform.system(), platform.release())
print("Prosesor    :", platform.processor() or "(tidak diketahui)")
print("pandas      :", pd.__version__, "| numpy:", np.__version__)

## Sel 4. Algoritma Kruskal dan Himpunan Terpisah (Disjoint Set)
**Gagasan (analogi kabel):** urutkan semua jalur dari yang termurah, lalu pasang jalur satu per satu, kecuali jalur yang membuat jaringan melingkar (siklus).
`HimpunanTerpisah` adalah alat untuk menjawab satu pertanyaan: "apakah dua kota sudah tersambung?"

**Alat bantu bawaan Python yang dipakai:** `sorted()` untuk mengurutkan sisi, biaya O(m log m). Logika memilih sisi dan mendeteksi siklus ditulis sendiri.
**Sumber:** *union by rank* dan kompresi lintasan (*path compression*) dibahas pada Sanders *et al.* (2019, Bagian 11.4). Kode memakai *path halving*, varian kompresi lintasan yang dipilih penulis dan tidak ditemukan pada bagian tersebut. Kode bukan salinan dari repositori atau forum tertentu.

In [ ]:
# ===== SEL 4: KRUSKAL DAN HIMPUNAN TERPISAH =====
class HimpunanTerpisah:
    """Disjoint Set (Union-Find) dengan union by rank dan path halving."""

    def __init__(self, jumlah_simpul):
        self.induk = list(range(jumlah_simpul))  # awalnya induk tiap simpul adalah dirinya
        self.peringkat = [0] * jumlah_simpul     # perkiraan tinggi pohon

    def cari_akar(self, simpul):
        """Mencari akar (ketua) himpunan dari simpul."""
        while self.induk[simpul] != simpul:
            self.induk[simpul] = self.induk[self.induk[simpul]]  # path halving
            simpul = self.induk[simpul]
        return simpul

    def gabung(self, simpul_a, simpul_b):
        """Gabungkan himpunan kedua simpul. False jika sudah satu himpunan (siklus)."""
        akar_a = self.cari_akar(simpul_a)
        akar_b = self.cari_akar(simpul_b)
        if akar_a == akar_b:
            return False
        if self.peringkat[akar_a] < self.peringkat[akar_b]:  # pohon pendek digantung ke pohon tinggi
            akar_a, akar_b = akar_b, akar_a
        self.induk[akar_b] = akar_a
        if self.peringkat[akar_a] == self.peringkat[akar_b]:
            self.peringkat[akar_a] += 1
        return True


def kruskal(jumlah_simpul, daftar_sisi):
    """Mengembalikan (sisi_mst, total_bobot). ValueError jika graf tidak terhubung."""
    # Urut menaik menurut bobot; bila sama, simpul terkecil dulu, lalu simpul terbesar.
    sisi_terurut = sorted(daftar_sisi, key=lambda sisi: (sisi[2], min(sisi[0], sisi[1]), max(sisi[0], sisi[1])))

    himpunan = HimpunanTerpisah(jumlah_simpul)
    sisi_mst = []
    total_bobot = 0

    for simpul_a, simpul_b, bobot in sisi_terurut:  # dari sisi paling ringan
        if himpunan.gabung(simpul_a, simpul_b):  # terima jika belum terhubung (tanpa siklus)
            sisi_mst.append((simpul_a, simpul_b, bobot))
            total_bobot += bobot
            if len(sisi_mst) == jumlah_simpul - 1:  # MST sudah lengkap
                break

    if len(sisi_mst) != jumlah_simpul - 1:
        raise ValueError("Graf tidak terhubung")
    return sisi_mst, total_bobot

## Sel 5. Algoritma Prim
**Gagasan (analogi kabel):** mulai dari satu kota, lalu tumbuhkan jaringan. Setiap kali, ambil tawaran jalur **termurah** yang keluar dari jaringan. Jika kota di ujungnya sudah tersambung, buang (akan melingkar). Jika belum, pasang dan tambahkan tawaran jalur baru dari kota itu.

**Alat bantu bawaan Python yang dipakai:** `heapq` sebagai antrean prioritas, biaya O(log m) per operasi.
**Varian *lazy*:** `heapq` tidak punya operasi *decrease-key*, jadi tawaran yang usang dibiarkan di antrean lalu dibuang saat keluar. Antrean bisa berisi sampai O(m) elemen, sehingga waktunya O(m log m), yang sama dengan O(m log n).
**Aturan pemutus seri:** isi antrean adalah (bobot, simpul_kecil, simpul_besar), jadi tawaran yang sama murah diambil menurut simpul terkecil lalu terbesar, sama dengan Kruskal dan Borůvka. Ujung yang sudah ada di pohon menjadi `asal`, ujung satunya menjadi `tujuan`.
**Representasi graf:** daftar ketetanggaan (sesuai batasan Bab 1), dibuat oleh `buat_daftar_tetangga`.

In [ ]:
# ===== SEL 5: PRIM =====
def buat_daftar_tetangga(jumlah_simpul, daftar_sisi):
    """Ubah daftar sisi (simpul_a, simpul_b, bobot) menjadi daftar ketetanggaan."""
    daftar_tetangga = [[] for _ in range(jumlah_simpul)]
    for simpul_a, simpul_b, bobot in daftar_sisi:
        daftar_tetangga[simpul_a].append((simpul_b, bobot))
        daftar_tetangga[simpul_b].append((simpul_a, bobot))  # graf tak berarah
    return daftar_tetangga


def prim(jumlah_simpul, daftar_tetangga, simpul_awal=0):
    """Mengembalikan (sisi_mst, total_bobot). ValueError jika graf tidak terhubung."""
    sudah_masuk = [False] * jumlah_simpul  # simpul yang sudah ada di pohon
    sisi_mst = []
    total_bobot = 0

    # Pohon dimulai dari satu simpul; masukkan semua sisi yang keluar darinya.
    sudah_masuk[simpul_awal] = True
    antrean = []
    for simpul_tetangga, bobot in daftar_tetangga[simpul_awal]:
        heapq.heappush(antrean, (bobot, min(simpul_awal, simpul_tetangga), max(simpul_awal, simpul_tetangga)))

    while antrean and len(sisi_mst) < jumlah_simpul - 1:
        bobot, kecil, besar = heapq.heappop(antrean)  # sisi termurah yang menyentuh pohon
        if sudah_masuk[kecil] and sudah_masuk[besar]:  # kedua ujung sudah di pohon: siklus
            continue
        asal, tujuan = (kecil, besar) if sudah_masuk[kecil] else (besar, kecil)  # tujuan = ujung baru
        sudah_masuk[tujuan] = True  # simpul baru bergabung ke pohon
        sisi_mst.append((asal, tujuan, bobot))
        total_bobot += bobot
        for simpul_tetangga, bobot_baru in daftar_tetangga[tujuan]:
            if not sudah_masuk[simpul_tetangga]:
                heapq.heappush(antrean, (bobot_baru, min(tujuan, simpul_tetangga), max(tujuan, simpul_tetangga)))

    if len(sisi_mst) != jumlah_simpul - 1:
        raise ValueError("Graf tidak terhubung")
    return sisi_mst, total_bobot

## Sel 6. Algoritma Borůvka (pembanding ketiga)
**Gagasan (analogi kabel):** semua kelompok kota bergerak **serentak**. Dalam satu putaran, tiap kelompok mencatat jalur termurah yang keluar dari kelompoknya, lalu semua pilihan dipasang sekaligus. Ulangi sampai tinggal satu kelompok. Jumlah kelompok menyusut sedikitnya separuh per putaran, sehingga putarannya paling banyak log₂ n.

**Aturan pemutus seri:** `kunci_sisi` membandingkan bobot, lalu simpul terkecil, lalu simpul terbesar (sama dengan urutan Kruskal dan isi antrean Prim), sesuai batasan Bab 1.
**Himpunan terpisah:** memakai `HimpunanTerpisah` dari Sel 4 (tidak disalin ulang).

In [ ]:
# ===== SEL 6: BORŮVKA =====
def kunci_sisi(sisi):
    """Kunci pembanding sisi: bobot, lalu simpul terkecil, lalu simpul terbesar (sama seperti Kruskal)."""
    return (sisi[2], min(sisi[0], sisi[1]), max(sisi[0], sisi[1]))


def boruvka(jumlah_simpul, daftar_sisi):
    """Mengembalikan (sisi_mst, total_bobot). ValueError jika graf tidak terhubung."""
    himpunan = HimpunanTerpisah(jumlah_simpul)
    sisi_mst = []
    total_bobot = 0
    jumlah_komponen = jumlah_simpul  # awalnya tiap simpul adalah satu komponen

    while jumlah_komponen > 1:
        termurah = {}  # akar komponen -> sisi termurah yang keluar dari komponen itu
        for simpul_a, simpul_b, bobot in daftar_sisi:
            akar_a = himpunan.cari_akar(simpul_a)
            akar_b = himpunan.cari_akar(simpul_b)
            if akar_a == akar_b:  # sisi di dalam satu komponen, abaikan
                continue
            sisi = (simpul_a, simpul_b, bobot)
            for akar in (akar_a, akar_b):  # sisi ini calon pilihan kedua komponen
                if akar not in termurah or kunci_sisi(sisi) < kunci_sisi(termurah[akar]):
                    termurah[akar] = sisi

        if not termurah:  # tidak ada sisi keluar: graf terpisah
            raise ValueError("Graf tidak terhubung")

        for simpul_a, simpul_b, bobot in termurah.values():  # gabungkan semua pilihan
            if himpunan.gabung(simpul_a, simpul_b):  # False jika sisi itu sudah dipasang
                sisi_mst.append((simpul_a, simpul_b, bobot))
                total_bobot += bobot
                jumlah_komponen -= 1

    return sisi_mst, total_bobot

## Sel 7. Uji kasus kecil (dicek dengan hitungan tangan)
Syarat dosen: **minimal 3 kasus uji kecil yang bisa dicek manual.** Di bawah ada 3 kasus graf terhubung dan 1 kasus graf tak terhubung. **Setiap kasus dijalankan pada ketiga algoritma**, sehingga syaratnya terpenuhi baik jika dosen memaksudkan 3 kasus total maupun 3 kasus per algoritma.

Jawaban yang benar di bawah **ditulis dari hitungan tangan, bukan dari keluaran program.**

> **PENTING: ini draf hitungan tangan. Hitung ulang sendiri di kertas sebelum dikumpulkan**, lalu pastikan kamu bisa menjelaskan tiap langkahnya saat tanya jawab. Nomor kota: Maros = 0, Makassar = 1, Gowa = 2, Takalar = 3.

### Hitungan tangan (cara Kruskal)
**Kasus 1 (4 kota biasa, ada satu jalur yang ditolak).** Jalur: Gowa–Takalar 4, Maros–Gowa 3, Makassar–Takalar 5, Maros–Makassar 1, Makassar–Gowa 2.
Urut termurah: Maros–Makassar 1, Makassar–Gowa 2, Maros–Gowa 3, Gowa–Takalar 4, Makassar–Takalar 5.
Ambil 1 (terima). Ambil 2 (terima). Maros–Gowa 3: Maros dan Gowa sudah tersambung lewat Makassar, jadi **ditolak (siklus)**. Gowa–Takalar 4: Takalar baru, terima. Sudah 3 kabel = n − 1, selesai.
**Total = 1 + 2 + 4 = 7.**

**Kasus 2 (segitiga, 3 kota).** Jalur: Maros–Makassar 2, Makassar–Gowa 3, Maros–Gowa 1.
Urut: Maros–Gowa 1, Maros–Makassar 2, Makassar–Gowa 3. Terima 1, terima 2, lalu Makassar–Gowa 3 **ditolak** karena keduanya sudah tersambung lewat Maros.
**Total = 1 + 2 = 3**, dengan 2 kabel.

**Kasus 3 (bobot kembar).** Jalur: Maros–Makassar 2, Makassar–Gowa 2, Gowa–Takalar 2, Takalar–Maros 2, dan diagonal Makassar–Takalar 5. Empat jalur biaya 2 membentuk lingkaran, jadi hanya tiga yang boleh dipasang.
Urutan pemeriksaan menurut aturan pemutus seri (bobot, lalu simpul terkecil, lalu simpul terbesar): Maros–Makassar (0, 1), Maros–Takalar (0, 3), Makassar–Gowa (1, 2), Gowa–Takalar (2, 3), lalu diagonal 5. Tiga yang pertama diterima. Gowa–Takalar **ditolak** karena Gowa dan Takalar sudah tersambung lewat Makassar dan Maros. Diagonal biaya 5 tidak diperlukan.
**Total = 2 + 2 + 2 = 6**, dengan MST unik {Maros–Makassar, Maros–Takalar, Makassar–Gowa}. Karena aturan pemutus seri sama pada ketiga algoritma, **total dan himpunan sisi** dicek.

**Kasus 4 (graf tak terhubung).** Jalur: Maros–Makassar 1 dan Gowa–Takalar 2. Maros dan Makassar tidak punya jalan ke Gowa dan Takalar, jadi **tidak ada MST**: ketiga algoritma harus menolak dengan `ValueError`.

In [ ]:
# ===== SEL 7: UJI KASUS KECIL =====
MAROS, MAKASSAR, GOWA, TAKALAR = 0, 1, 2, 3  # nomor simpul tiap kota
nama_kota = ["Maros", "Makassar", "Gowa", "Takalar"]

# Tiap kasus: jawaban benar berasal dari hitungan tangan (lihat sel penjelasan di atas).
kasus_uji = [
    {"nama": "Kasus 1: 4 kota biasa (satu jalur ditolak)", "jumlah_simpul": 4,
     "daftar_sisi": [(GOWA, TAKALAR, 4), (MAROS, GOWA, 3), (MAKASSAR, TAKALAR, 5),
                     (MAROS, MAKASSAR, 1), (MAKASSAR, GOWA, 2)],
     "total_benar": 7,
     "sisi_benar": {(MAROS, MAKASSAR, 1), (MAKASSAR, GOWA, 2), (GOWA, TAKALAR, 4)}},
    {"nama": "Kasus 2: segitiga 3 kota", "jumlah_simpul": 3,
     "daftar_sisi": [(MAROS, MAKASSAR, 2), (MAKASSAR, GOWA, 3), (MAROS, GOWA, 1)],
     "total_benar": 3,
     "sisi_benar": {(MAROS, GOWA, 1), (MAROS, MAKASSAR, 2)}},
    {"nama": "Kasus 3: bobot kembar (total dan himpunan sisi)", "jumlah_simpul": 4,
     "daftar_sisi": [(MAROS, MAKASSAR, 2), (MAKASSAR, GOWA, 2), (GOWA, TAKALAR, 2),
                     (TAKALAR, MAROS, 2), (MAKASSAR, TAKALAR, 5)],
     "total_benar": 6,
     "sisi_benar": {(MAROS, MAKASSAR, 2), (MAROS, TAKALAR, 2), (MAKASSAR, GOWA, 2)}},  # unik karena aturan pemutus seri
]

kasus_tak_terhubung = {"nama": "Kasus 4: graf tak terhubung (harus ValueError)", "jumlah_simpul": 4,
                       "daftar_sisi": [(MAROS, MAKASSAR, 1), (GOWA, TAKALAR, 2)]}


def jalankan_tiga_algoritma(jumlah_simpul, daftar_sisi):
    """Menjalankan ketiga algoritma pada graf yang sama. Mengembalikan {nama: (sisi_mst, total)}."""
    return {
        "Kruskal": kruskal(jumlah_simpul, daftar_sisi),
        "Prim": prim(jumlah_simpul, buat_daftar_tetangga(jumlah_simpul, daftar_sisi)),
        "Borůvka": boruvka(jumlah_simpul, daftar_sisi),
    }


def rapikan(sisi_mst):
    """Ubah sisi menjadi (kecil, besar, bobot) supaya arah sisi tidak memengaruhi perbandingan."""
    return {(min(a, b), max(a, b), w) for a, b, w in sisi_mst}


jumlah_lolos = 0
for kasus in kasus_uji:
    hasil = jalankan_tiga_algoritma(kasus["jumlah_simpul"], kasus["daftar_sisi"])
    for algoritma, (sisi_mst, total) in hasil.items():
        assert total == kasus["total_benar"], f"{kasus['nama']}: total {algoritma} = {total}, seharusnya {kasus['total_benar']}"
        assert len(sisi_mst) == kasus["jumlah_simpul"] - 1, f"{kasus['nama']}: jumlah sisi {algoritma} salah"
        if kasus["sisi_benar"] is not None:
            assert rapikan(sisi_mst) == rapikan(kasus["sisi_benar"]), f"{kasus['nama']}: himpunan sisi {algoritma} salah"
    jumlah_lolos += 1
    print(f"[LOLOS] {kasus['nama']} | total = {kasus['total_benar']} | Kruskal, Prim, Borůvka sama")

# Kasus 4: ketiga algoritma harus menolak graf tak terhubung
for algoritma, fungsi in [("Kruskal", lambda n, s: kruskal(n, s)),
                          ("Prim", lambda n, s: prim(n, buat_daftar_tetangga(n, s))),
                          ("Borůvka", lambda n, s: boruvka(n, s))]:
    try:
        fungsi(kasus_tak_terhubung["jumlah_simpul"], kasus_tak_terhubung["daftar_sisi"])
        raise AssertionError(f"{algoritma} seharusnya melempar ValueError")
    except ValueError:
        pass
print(f"[LOLOS] {kasus_tak_terhubung['nama']} | ketiganya melempar ValueError")

JUMLAH_KASUS_UJI = jumlah_lolos + 1  # kasus graf terhubung + satu kasus graf tak terhubung
print(f"\nSemua {JUMLAH_KASUS_UJI} kasus uji lolos pada ketiga algoritma ({jumlah_lolos} kasus graf terhubung dan 1 kasus galat).")

## Sel 7B. Studi kasus nyata: jaringan kabel antar gedung Fakultas Teknik Unhas
Satu contoh penerapan pada **data tetap**, bukan graf acak, sehingga **tidak memakai seed** dan hasilnya sama setiap dijalankan. Datanya dibaca dari `data/studi_kasus_gedung_unhas.csv` (kolom: gedung_a, gedung_b, jarak_m, keterangan). Simpul diberi nomor tetap menurut urutan abjad nama gedung, sehingga aturan pemutus seri pada Batasan 7 tetap berlaku.

> **PENTING:** selama kolom `keterangan` masih berisi **ILUSTRASI**, angkanya adalah contoh karangan, bukan hasil ukur. Ganti dengan jarak yang Anda ukur sendiri (misalnya dari peta daring), catat sumber dan tanggalnya di Bab 4, lalu hapus kata ILUSTRASI. Graf ini hanya 5 simpul, jadi dipakai untuk menunjukkan kebenaran dan penerapan, **bukan** untuk mengukur waktu.

In [ ]:
# ===== SEL 7B: STUDI KASUS NYATA (JARINGAN ANTAR GEDUNG) =====
BERKAS_KASUS = os.path.join(os.getcwd(), "data", "studi_kasus_gedung_unhas.csv")
with open(BERKAS_KASUS, newline="", encoding="utf-8") as berkas:
    baris_kasus = list(csv.DictReader(berkas))

nama_gedung = sorted({b[k] for b in baris_kasus for k in ("gedung_a", "gedung_b")})
nomor_gedung = {nama: i for i, nama in enumerate(nama_gedung)}  # nomor tetap menurut abjad
sisi_kasus = [(nomor_gedung[b["gedung_a"]], nomor_gedung[b["gedung_b"]], int(b["jarak_m"])) for b in baris_kasus]

hasil_kasus = jalankan_tiga_algoritma(len(nama_gedung), sisi_kasus)
total_kasus = {a: t for a, (_, t) in hasil_kasus.items()}
assert len(set(total_kasus.values())) == 1, f"total bobot berbeda: {total_kasus}"
himpunan_kasus = {a: rapikan(s) for a, (s, _) in hasil_kasus.items()}
assert len({frozenset(h) for h in himpunan_kasus.values()}) == 1, "himpunan sisi ketiga algoritma berbeda"

sisi_mst_kasus, total_mst_kasus = hasil_kasus["Kruskal"]
print(f"Studi kasus: {len(nama_gedung)} gedung, {len(sisi_kasus)} jalur kabel yang mungkin")
for a, b, w in sorted(rapikan(sisi_mst_kasus), key=lambda s: (s[2], s[0], s[1])):
    print(f"  {nama_gedung[a]} - {nama_gedung[b]}: {w} m")
print(f"Total kabel minimum: {total_mst_kasus} m (Kruskal, Prim, dan Borůvka sama)")
if any(b["keterangan"].upper().startswith("ILUSTRASI") for b in baris_kasus):
    print("PERINGATAN: data masih berlabel ILUSTRASI. Ganti dengan hasil ukur sebelum disebut data nyata.")


## Sel 8. Pembuat graf acak (seed tetap, selalu terhubung)
**Cara kerja dua tahap:**
1. **Pohon acak.** Simpul 1 sampai n−1 masing-masing disambungkan ke satu simpul acak bernomor lebih kecil. Ini menghasilkan n−1 sisi dan graf **pasti terhubung**.
2. **Tambah sisi acak** sampai mencapai jumlah sisi target, dengan melewati pasangan yang sudah ada (**tanpa sisi ganda dan tanpa loop**, sesuai batasan Bab 1).

Pengacak dibuat sendiri dengan `random.Random(seed)`, bukan pengacak global, sehingga seed yang sama selalu menghasilkan graf yang sama. Urutan sisi diacak supaya tidak ada urutan istimewa.

In [ ]:
# ===== SEL 8: PEMBUAT GRAF ACAK =====
def buat_graf(jumlah_simpul, jumlah_sisi, bobot_maks, seed):
    """Graf acak terhubung tanpa sisi ganda dan tanpa loop. Mengembalikan daftar sisi."""
    maks_sisi = jumlah_simpul * (jumlah_simpul - 1) // 2
    if not (jumlah_simpul - 1 <= jumlah_sisi <= maks_sisi):
        raise ValueError(f"jumlah_sisi harus antara {jumlah_simpul - 1} dan {maks_sisi}")
    pengacak = random.Random(seed)
    sudah_ada = set()
    daftar_sisi = []

    # Tahap 1: pohon acak, supaya graf pasti terhubung.
    for simpul in range(1, jumlah_simpul):
        simpul_lain = pengacak.randrange(simpul)
        sudah_ada.add((simpul_lain, simpul))
        daftar_sisi.append((simpul_lain, simpul, pengacak.randint(1, bobot_maks)))

    # Tahap 2: tambah sisi acak sampai jumlahnya cukup.
    while len(daftar_sisi) < jumlah_sisi:
        a = pengacak.randrange(jumlah_simpul)
        b = pengacak.randrange(jumlah_simpul)
        if a == b:
            continue
        if a > b:
            a, b = b, a
        if (a, b) in sudah_ada:
            continue
        sudah_ada.add((a, b))
        daftar_sisi.append((a, b, pengacak.randint(1, bobot_maks)))

    pengacak.shuffle(daftar_sisi)
    return daftar_sisi

## Sel 9. Periksa pembuat graf (seed tetap, terhubung, kepadatan)
Tiga pemeriksaan sebelum graf dipakai untuk eksperimen: (1) seed yang sama menghasilkan graf yang sama, (2) graf selalu terhubung dan tidak punya sisi ganda atau loop, (3) jumlah sisi sesuai skenario jarang dan padat.

In [ ]:
# ===== SEL 9: PERIKSA PEMBUAT GRAF =====
def graf_terhubung(jumlah_simpul, daftar_sisi):
    himpunan = HimpunanTerpisah(jumlah_simpul)
    for a, b, _ in daftar_sisi:
        himpunan.gabung(a, b)
    return len({himpunan.cari_akar(i) for i in range(jumlah_simpul)}) == 1


# (1) seed sama -> graf sama; seed beda -> graf beda
g1 = buat_graf(50, 120, 100, seed=SEED)
g2 = buat_graf(50, 120, 100, seed=SEED)
g3 = buat_graf(50, 120, 100, seed=SEED + 1)
assert g1 == g2 and g1 != g3
print("Seed tetap: seed sama menghasilkan graf identik, seed beda menghasilkan graf lain.")

# (2) terhubung, tanpa sisi ganda dan loop; (3) jumlah sisi sesuai skenario
for nama, s in SKENARIO.items():
    n = s["ukuran"][0]
    m = s["jumlah_sisi"](n)
    graf = buat_graf(n, m, s["bobot_maks"], seed=SEED)
    pasangan = {(min(a, b), max(a, b)) for a, b, _ in graf}
    assert graf_terhubung(n, graf), "graf tidak terhubung"
    assert len(pasangan) == len(graf) == m, "ada sisi ganda atau jumlah sisi salah"
    assert all(a != b for a, b, _ in graf), "ada loop"
    kepadatan = m / (n * (n - 1) / 2)
    print(f"{nama:14s}: n = {n:5d}, m = {m:6d}, kepadatan = {kepadatan:.4f} | terhubung, tanpa sisi ganda atau loop")

## Sel 10. Verifikasi tambahan dengan `networkx` (opsional)
`networkx` **hanya dipakai sebagai pembanding**, tidak di dalam algoritma, sesuai aturan dosen (pustaka boleh sebagai pembanding). Sel ini membandingkan total bobot ketiga algoritma dengan `networkx` pada 300 graf acak kecil. Ini tambahan di luar uji kasus kecil pada Sel 7, yang jawabannya dari hitungan tangan.

In [ ]:
# ===== SEL 10: VERIFIKASI TAMBAHAN DENGAN NETWORKX (OPSIONAL) =====
try:
    import networkx as nx
except ImportError:
    nx = None
    print("networkx tidak terpasang, sel ini dilewati (opsional).")

if nx is not None:
    pengacak = random.Random(SEED)
    jumlah_graf = 300
    for _ in range(jumlah_graf):
        n = pengacak.randint(2, 40)
        m = pengacak.randint(n - 1, min(n * (n - 1) // 2, 4 * n))
        graf = buat_graf(n, m, pengacak.choice([3, 10, 1000]), seed=pengacak.randint(0, 10**9))
        G = nx.Graph()
        G.add_nodes_from(range(n))
        G.add_weighted_edges_from(graf)
        total_referensi = nx.minimum_spanning_tree(G).size(weight="weight")
        for algoritma, (sisi_mst, total) in jalankan_tiga_algoritma(n, graf).items():
            assert total == total_referensi and len(sisi_mst) == n - 1, f"{algoritma} berbeda dari networkx"
    print(f"Lolos: total bobot Kruskal, Prim, dan Borůvka sama dengan networkx pada {jumlah_graf} graf acak.")

# --- Cek: file .py di folder Kruskal, Prim, dan Borůvka memberi hasil yang sama dengan kode di notebook ini ---
import importlib.util

folder_induk = os.path.dirname(FOLDER_DASAR)  # folder CODING
berkas_py = {
    "Kruskal": os.path.join(folder_induk, "Kruskal", "kruskal.py"),
    "Prim": os.path.join(folder_induk, "Prim", "prim.py"),
    "Borůvka": os.path.join(folder_induk, "Borůvka", "boruvka.py"),
}
if all(os.path.exists(p) for p in berkas_py.values()):
    def muat_modul(nama, jalur):
        spesifikasi = importlib.util.spec_from_file_location(nama, jalur)
        modul = importlib.util.module_from_spec(spesifikasi)
        spesifikasi.loader.exec_module(modul)
        return modul

    sys.path.insert(0, os.path.dirname(berkas_py["Kruskal"]))  # supaya boruvka.py menemukan kruskal.py
    modul_kruskal = muat_modul("kruskal_berkas", berkas_py["Kruskal"])
    modul_prim = muat_modul("prim_berkas", berkas_py["Prim"])
    modul_boruvka = muat_modul("boruvka_berkas", berkas_py["Borůvka"])
    pengacak = random.Random(SEED)
    for _ in range(100):
        n = pengacak.randint(2, 60)
        graf = buat_graf(n, pengacak.randint(n - 1, min(n * (n - 1) // 2, 4 * n)), pengacak.choice([3, 1000]), seed=pengacak.randint(0, 10**9))
        assert kruskal(n, graf) == modul_kruskal.kruskal(n, graf), "kruskal.py berbeda dari notebook"
        assert prim(n, buat_daftar_tetangga(n, graf)) == modul_prim.prim(n, modul_prim.buat_daftar_tetangga(n, graf)), "prim.py berbeda dari notebook"
        assert boruvka(n, graf) == modul_boruvka.boruvka(n, graf), "boruvka.py berbeda dari notebook"
    print("Lolos: file .py di folder Kruskal, Prim, dan Borůvka memberi hasil yang sama dengan kode di notebook ini (100 graf acak).")
else:
    print("Folder Kruskal, Prim, atau Borůvka tidak ditemukan di samping folder ini, pencocokan dilewati.")

## Sel 11. Fungsi pengukuran waktu
**Aturan pengukuran:**
- Waktu diukur dengan `time.perf_counter()` dan **hanya** mengukur pemanggilan algoritma. Waktu membuat graf tidak ikut dihitung.
- Ketiga algoritma dijalankan pada **graf yang sama** supaya perbandingannya adil.
- Setiap algoritma dijalankan sekali **pemanasan** (tidak dicatat) sebelum diukur.
- **Prim** butuh daftar ketetanggaan, sedangkan Kruskal dan Borůvka memakai daftar sisi. Karena itu waktu Prim dicatat dua kali: **"Prim"** (hanya inti algoritma) dan **"Prim (+konversi)"** (ditambah waktu membuat daftar ketetanggaan). Keduanya dilaporkan agar perbandingannya jujur.
- Total bobot ketiga algoritma harus sama pada setiap ulangan. Jika berbeda, program berhenti.

In [ ]:
# ===== SEL 11: FUNGSI PENGUKURAN WAKTU =====
import gc

ALGORITMA_UTAMA = ["Kruskal", "Prim", "Borůvka"]
ALGORITMA_CSV = ["Kruskal", "Prim", "Prim (+konversi)", "Borůvka"]


def ukur(fungsi, *argumen):
    """Menjalankan fungsi sekali. Mengembalikan (hasil, waktu_detik)."""
    gc.collect()
    mulai = time.perf_counter()
    hasil = fungsi(*argumen)
    selesai = time.perf_counter()
    return hasil, selesai - mulai


def ukur_satu_ulangan(jumlah_simpul, daftar_sisi):
    """Mengukur ketiga algoritma pada satu graf. Mengembalikan {nama algoritma: (waktu, total_bobot)}."""
    daftar_tetangga, waktu_konversi = ukur(buat_daftar_tetangga, jumlah_simpul, daftar_sisi)
    (_, total_kruskal), waktu_kruskal = ukur(kruskal, jumlah_simpul, daftar_sisi)
    (_, total_prim), waktu_prim = ukur(prim, jumlah_simpul, daftar_tetangga)
    (_, total_boruvka), waktu_boruvka = ukur(boruvka, jumlah_simpul, daftar_sisi)
    assert total_kruskal == total_prim == total_boruvka, \
        f"Total bobot berbeda: Kruskal {total_kruskal}, Prim {total_prim}, Borůvka {total_boruvka}"
    return {
        "Kruskal": (waktu_kruskal, total_kruskal),
        "Prim": (waktu_prim, total_prim),
        "Prim (+konversi)": (waktu_prim + waktu_konversi, total_prim),
        "Borůvka": (waktu_boruvka, total_boruvka),
    }


# Contoh singkat satu pengukuran pada graf kecil
contoh_graf = buat_graf(1000, 3000, BOBOT_MAKS_BIASA, seed=SEED)
for algoritma, (detik, total) in ukur_satu_ulangan(1000, contoh_graf).items():
    print(f"{algoritma:18s}: {detik * 1000:8.2f} ms | total bobot = {total}")

## Sel 12. Jalankan eksperimen dan simpan ke CSV
Untuk setiap skenario dan setiap ukuran: buat graf (seed tetap), lakukan pemanasan, lalu ukur sebanyak `ULANGAN` kali. **Data mentah tiap ulangan** disimpan ke `hasil/eksperimen_mentah.csv`, sehingga rata-rata dan simpangan baku bisa dihitung ulang kapan saja.

Proses ini memakan sekitar 1 sampai 3 menit tergantung komputer, terutama pada graf padat. Kemajuannya dicetak di bawah.

In [ ]:
# ===== SEL 12: JALANKAN EKSPERIMEN DAN SIMPAN KE CSV =====
BERKAS_MENTAH = os.path.join(FOLDER_HASIL, "eksperimen_mentah.csv")
KOLOM = ["skenario", "n", "m", "algoritma", "ulangan", "waktu_detik", "total_bobot"]

waktu_mulai_eksperimen = time.perf_counter()
with open(BERKAS_MENTAH, "w", newline="", encoding="utf-8") as berkas:
    penulis = csv.writer(berkas)
    penulis.writerow(KOLOM)
    for nama_skenario, s in SKENARIO.items():
        for n in s["ukuran"]:
            m = s["jumlah_sisi"](n)
            graf_tetap = buat_graf(n, m, s["bobot_maks"], seed=SEED)  # dipakai bila ULANG_GRAF_BARU = False
            # pemanasan: hasilnya tidak dicatat
            ukur_satu_ulangan(n, graf_tetap)
            for ulangan in range(1, ULANGAN + 1):
                graf = buat_graf(n, m, s["bobot_maks"], seed=SEED + ulangan) if ULANG_GRAF_BARU else graf_tetap
                hasil = ukur_satu_ulangan(n, graf)
                for algoritma in ALGORITMA_CSV:
                    detik, total = hasil[algoritma]
                    penulis.writerow([nama_skenario, n, m, algoritma, ulangan, f"{detik:.6f}", total])
            print(f"selesai: {nama_skenario:14s} n = {n:6d}, m = {m:7d}")

lama = time.perf_counter() - waktu_mulai_eksperimen
print(f"\nEksperimen selesai dalam {lama:.1f} detik. Data mentah: {BERKAS_MENTAH}")

## Sel 13. Ringkasan: rata-rata dan simpangan baku
Untuk tiap kombinasi skenario, ukuran, dan algoritma dihitung **rata-rata** dan **simpangan baku** dari semua ulangan. Simpangan baku yang dipakai adalah simpangan baku **sampel** (dibagi n − 1, sama dengan `statistics.stdev`). Hasilnya disimpan ke `hasil/ringkasan.csv`.

**Catatan:** dengan `ULANG_GRAF_BARU = False`, kelima ulangan memakai graf yang sama, sehingga simpangan baku mengukur **gangguan waktu pengukuran**, bukan variasi antar-graf.


In [ ]:
# ===== SEL 13: RINGKASAN RATA-RATA DAN SIMPANGAN BAKU =====
data_mentah = pd.read_csv(BERKAS_MENTAH)
ringkasan = (data_mentah
             .groupby(["skenario", "n", "m", "algoritma"], sort=False)["waktu_detik"]
             .agg(rata_rata="mean", simpangan_baku="std", jumlah_ulangan="count")
             .reset_index())
BERKAS_RINGKASAN = os.path.join(FOLDER_HASIL, "ringkasan.csv")
ringkasan.to_csv(BERKAS_RINGKASAN, index=False)

# Tampilan dalam milidetik agar mudah dibaca
tampil = ringkasan.copy()
tampil["rata_rata (ms)"] = (tampil["rata_rata"] * 1000).round(3)
tampil["simpangan_baku (ms)"] = (tampil["simpangan_baku"] * 1000).round(3)
for nama_skenario in SKENARIO:
    print(f"\n=== Skenario: {nama_skenario} ({SKENARIO[nama_skenario]['keterangan']}) ===")
    bagian = tampil[tampil["skenario"] == nama_skenario]
    print(bagian.pivot(index=["n", "m"], columns="algoritma", values="rata_rata (ms)")[ALGORITMA_CSV].to_string())
print("\nRingkasan lengkap disimpan di:", BERKAS_RINGKASAN)

## Sel 14. Kurva teori dan grafik waktu
**Kurva teori** berasal dari kompleksitas yang diturunkan di Bab 3 laporan:

| Algoritma | Bentuk teori yang digambar | Alasan singkat |
|---|---|---|
| Kruskal | c · m · log₂ m | Pengurutan sisi O(m log m) mendominasi |
| Prim | c · m · log₂ n | Operasi *heap* O(log) untuk O(m) tawaran |
| Borůvka | c · m · log₂ n | Paling banyak log₂ n putaran, tiap putaran memindai m sisi |

Konstanta **c dicocokkan** dengan metode kuadrat terkecil melalui titik asal, karena teori hanya menyatakan **bentuk** kurva, bukan waktu mutlaknya. Yang diuji adalah apakah **bentuk** titik ukur mengikuti bentuk kurva teori.
> Jika turunan di Bab 3 milikmu berbeda dari tabel ini, ubah isi `KURVA_TEORI` di sel di bawah supaya konsisten dengan laporan.

Grafik kiri memakai sumbu linear, grafik kanan memakai sumbu log-log (garis lurus pada log-log menunjukkan pola pangkat). Batang kesalahan menunjukkan satu simpangan baku.

In [ ]:
# ===== SEL 14: KURVA TEORI DAN GRAFIK WAKTU =====
KURVA_TEORI = {  # f(n, m): bentuk teori, tanpa konstanta
    "Kruskal": lambda n, m: m * math.log2(m),
    "Prim": lambda n, m: m * math.log2(n),
    "Borůvka": lambda n, m: m * math.log2(n),
}
WARNA = {"Kruskal": "tab:blue", "Prim": "tab:green", "Borůvka": "tab:red", "Prim (+konversi)": "tab:olive"}
BERKAS_GRAFIK = {}


def cocokkan_konstanta(nilai_teori, waktu_ukur):
    """Konstanta c terbaik (kuadrat terkecil melalui titik asal) untuk waktu = c * teori."""
    nilai_teori, waktu_ukur = np.array(nilai_teori, float), np.array(waktu_ukur, float)
    return float((nilai_teori * waktu_ukur).sum() / (nilai_teori ** 2).sum())


for nama_skenario in SKENARIO:
    data = ringkasan[ringkasan["skenario"] == nama_skenario]
    fig, (ax_linear, ax_log) = plt.subplots(1, 2, figsize=(14, 5))
    for algoritma in ALGORITMA_CSV:
        d = data[data["algoritma"] == algoritma].sort_values("m")
        garis = "o:" if algoritma == "Prim (+konversi)" else "o-"
        for ax in (ax_linear, ax_log):
            ax.errorbar(d["m"], d["rata_rata"], yerr=d["simpangan_baku"], fmt=garis,
                        color=WARNA[algoritma], capsize=3, label=f"{algoritma} (ukur)")
        if algoritma in KURVA_TEORI:
            teori = [KURVA_TEORI[algoritma](n, m) for n, m in zip(d["n"], d["m"])]
            c = cocokkan_konstanta(teori, d["rata_rata"])
            m_halus = np.linspace(d["m"].min(), d["m"].max(), 200)
            n_halus = np.interp(m_halus, d["m"], d["n"])
            kurva = [c * KURVA_TEORI[algoritma](n, m) for n, m in zip(n_halus, m_halus)]
            for ax in (ax_linear, ax_log):
                ax.plot(m_halus, kurva, "--", color=WARNA[algoritma], alpha=0.7, label=f"{algoritma} (teori)")
    ax_linear.set_title(f"Skenario {nama_skenario}: skala linear")
    ax_log.set_title(f"Skenario {nama_skenario}: skala log-log")
    ax_log.set_xscale("log"); ax_log.set_yscale("log")
    for ax in (ax_linear, ax_log):
        ax.set_xlabel("jumlah sisi (m)"); ax.set_ylabel("waktu (detik)"); ax.grid(alpha=0.3)
    ax_linear.legend(fontsize=8)
    fig.suptitle(f"Waktu eksekusi vs ukuran graf ({SKENARIO[nama_skenario]['keterangan']})")
    fig.tight_layout()
    berkas = os.path.join(FOLDER_GRAFIK, f"waktu_{nama_skenario}.png")
    fig.savefig(berkas, dpi=150, bbox_inches="tight")
    BERKAS_GRAFIK[nama_skenario] = berkas
    plt.show()
print("Grafik disimpan di:", FOLDER_GRAFIK)

## Sel 15. Pemeriksaan kesesuaian dengan teori (tabel rasio)
Jika waktu mengikuti bentuk teori, maka **waktu ÷ teori(n, m)** kira-kira **konstan** di semua ukuran. Tabel di bawah menunjukkan rasio itu, dinormalkan terhadap ukuran terkecil (nilai 1,00). Angka yang tetap dekat 1,00 berarti sesuai teori. Angka yang menyimpang jauh perlu dijelaskan di pembahasan (misalnya efek *cache*, atau waktu yang terlalu singkat pada ukuran kecil sehingga berisik). Hasil yang kurang bagus **tetap bagian dari penelitian** selama didokumentasikan dan dievaluasi.

In [ ]:
# ===== SEL 15: TABEL RASIO TERHADAP TEORI =====
baris_rasio = []
for nama_skenario in SKENARIO:
    for algoritma in ALGORITMA_UTAMA:
        d = (ringkasan[(ringkasan["skenario"] == nama_skenario) & (ringkasan["algoritma"] == algoritma)]
             .sort_values("m").reset_index(drop=True))
        teori = np.array([KURVA_TEORI[algoritma](n, m) for n, m in zip(d["n"], d["m"])])
        rasio = d["rata_rata"].to_numpy() / teori
        rasio = rasio / rasio[0]  # dinormalkan terhadap ukuran terkecil
        for n, m, r in zip(d["n"], d["m"], rasio):
            baris_rasio.append({"skenario": nama_skenario, "algoritma": algoritma, "n": n, "m": m, "rasio": round(float(r), 3)})
tabel_rasio = pd.DataFrame(baris_rasio)
tabel_rasio.to_csv(os.path.join(FOLDER_HASIL, "rasio_terhadap_teori.csv"), index=False)
for nama_skenario in SKENARIO:
    print(f"\n=== Rasio waktu/teori, skenario {nama_skenario} (ukuran terkecil = 1,00) ===")
    print(tabel_rasio[tabel_rasio["skenario"] == nama_skenario]
          .pivot(index=["n", "m"], columns="algoritma", values="rasio")[ALGORITMA_UTAMA].to_string())

## Sel 16. Pemeriksaan kelengkapan syarat Tugas 2
Sel ini memeriksa otomatis apakah syarat jumlah dan berkas sudah terpenuhi. Tanda **[OK]** berarti terpenuhi.

In [ ]:
# ===== SEL 16: PEMERIKSAAN KELENGKAPAN SYARAT =====
def periksa(nama, kondisi):
    print(("[OK]    " if kondisi else "[KURANG] ") + nama)
    return kondisi

semua = []
semua.append(periksa("Kruskal, Prim, dan Borůvka ada dan berjalan", all(callable(f) for f in (kruskal, prim, boruvka))))
semua.append(periksa(f"Minimal 3 kasus uji kecil (graf terhubung) lolos pada ketiga algoritma (ada {JUMLAH_KASUS_UJI - 1})", JUMLAH_KASUS_UJI - 1 >= 3))
semua.append(periksa(f"Seed tetap dipakai (SEED = {SEED})", isinstance(SEED, int)))
for nama_skenario, s in SKENARIO.items():
    semua.append(periksa(f"Skenario {nama_skenario}: minimal 5 ukuran berbeda ({len(set(s['ukuran']))} ukuran)", len(set(s["ukuran"])) >= 5))
semua.append(periksa(f"Minimal 5 ulangan per percobaan (ULANGAN = {ULANGAN})", ULANGAN >= 5))
semua.append(periksa("Setiap kombinasi punya >= 5 data (rata-rata dan simpangan baku valid)", (ringkasan["jumlah_ulangan"] >= 5).all()))
semua.append(periksa("CSV data mentah tersimpan", os.path.exists(BERKAS_MENTAH)))
semua.append(periksa("CSV ringkasan tersimpan", os.path.exists(BERKAS_RINGKASAN)))
semua.append(periksa("Grafik dengan kurva teori tersimpan untuk semua skenario", all(os.path.exists(p) for p in BERKAS_GRAFIK.values())))
semua.append(periksa("Pembanding ketiga (Borůvka) ikut diukur", "Borůvka" in set(data_mentah["algoritma"])))
print("\nHasil: SEMUA SYARAT KODE TERPENUHI" if all(semua) else "\nMasih ada yang kurang, lihat tanda [KURANG].")
print("Folder hasil:", FOLDER_DASAR)

## Sel 17. Catatan metodologi, sumber, dan laporan penggunaan AI

### Keputusan desain (tulis juga di Bab 3 dan Bab 4)
| Keputusan | Pilihan di notebook ini | Alasan |
|---|---|---|
| Alat bantu bawaan | `sorted()` (Kruskal), `heapq` (Prim) | Alat umum yang tidak menghasilkan MST. Logika memilih sisi dan mendeteksi siklus ditulis sendiri. Biayanya: pengurutan O(m log m), operasi *heap* O(log m) |
| Varian Prim | *Lazy* (tanpa *decrease-key*) | `heapq` tidak punya *decrease-key*. Antrean bisa berisi O(m) elemen, waktunya O(m log m) = O(m log n) |
| Representasi graf | Daftar sisi (Kruskal, Borůvka), daftar ketetanggaan (Prim) | Sesuai batasan Bab 1 |
| Arti "ulangan" | Mengulang pengukuran pada graf yang sama (`ULANG_GRAF_BARU = False`) | Mengukur gangguan waktu. Ubah ke True untuk ikut mengukur variasi antar-graf |
| Waktu konversi Prim | Dilaporkan dua kali (tanpa dan dengan konversi) | Agar perbandingan dengan Kruskal dan Borůvka adil |
| Pemutus seri bobot kembar | Bobot, lalu simpul terkecil, lalu simpul terbesar (`kunci_sisi` di Kruskal dan Borůvka, isi antrean di Prim) | Konsisten di ketiga algoritma, sehingga MST unik dan himpunan sisinya bisa dibandingkan |
| Simpangan baku | Simpangan baku sampel (dibagi n − 1) | Standar untuk sampel kecil |
| Pengukuran | `time.perf_counter()`, pemanasan sekali, hanya mengukur algoritma | Mengurangi derau dan efek pemanasan |
| Lingkungan | Satu mesin, satu proses, Python murni | Batasan Bab 1 |

### Batasan
- Waktu diukur pada satu lingkungan perangkat keras dan perangkat lunak (lihat Sel 3). Hasil mutlak berbeda di komputer lain, sedangkan **bentuk** pertumbuhannya yang dibandingkan dengan teori.
- Kompleksitas ruang hanya dianalisis secara teoretis (batasan Bab 1).
- Waktu Python murni mencakup beban interpreter, jadi konstanta jauh lebih besar daripada implementasi C++ dan sifatnya hanya perbandingan relatif.

### Sumber
- P. Sanders, K. Mehlhorn, M. Dietzfelbinger, R. Dementiev, *Sequential and Parallel Algorithms and Data Structures: The Basic Toolbox*, edisi ke-2, Springer, 2019, Bab 11: penyajian Kruskal, Prim, Borůvka, dan struktur data *disjoint set* (Bagian 11.2 sampai 11.4 dan 11.6).
- Kruskal (1956), Prim (1957), dan Nešetřil *et al.* (2001) untuk Borůvka, seperti di daftar pustaka Bab 1.
- Tidak ada kode yang disalin dari repositori GitHub, buku, atau forum. Draf kode dibuat dengan bantuan AI (lihat laporan di bawah).

### Laporan penggunaan AI (DRAF, sesuaikan dengan kenyataan sebelum dikumpulkan)
> **Isi bagian ini dengan jujur dan dengan kata-katamu sendiri.** Dosen mewajibkan pemakaian AI dilaporkan di lampiran, dan kamu harus bisa menjelaskan setiap baris kode serta setiap langkah penurunan kompleksitas saat tanya jawab. Jika kamu menulis ulang bagian kode dengan versimu sendiri, catat itu di sini.

| Butir | Isi (draf) |
|---|---|
| Alat AI yang dipakai | Claude (Anthropic) |
| Tujuan | Memahami konsep Kruskal, Prim, dan Borůvka; contoh penerapan; contoh implementasi; *debugging*; pembuatan kerangka eksperimen |
| Bagian pekerjaan yang dibantu | Draf kode Kruskal, Prim, Borůvka; kerangka notebook, pembuat graf, dan eksperimen; visualisasi HTML untuk belajar |
| Cara verifikasi | (1) Uji kasus kecil dengan hitungan tangan (Sel 7); (2) perbandingan total bobot dengan `networkx` pada 300 graf acak kecil; (3) pemeriksaan total bobot ketiga algoritma sama pada setiap ulangan eksperimen |
| Referensi | Setiap referensi di laporan sudah dicek keberadaannya ke sumber aslinya (**centang setelah kamu benar-benar memeriksa**) |
| Bagian yang kamu tulis atau ubah sendiri | (isi sendiri) |